# A/B 근거 반환·품질·지연 측정
**A**: RetroMAE CLS + dot product 검색 → 원문 선택.
**B**: 같은 dense 후보 → DeBERTa-v3 재정렬 → 원문 선택.

품질은 저장된 동일 후보를 공유한다. 지연은 캐시를 사용하지 않고 각 방법이 질문 임베딩부터 독립 실행한다.
기본값은 개발셋 종류별 5문항·1회 반복의 동작 검증이다. 본 평가 결과로 해석하지 않는다.

### 1. 환경과 변경 가능한 초기 조건

In [ ]:
from pathlib import Path
import sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'config/baselines.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config/baselines.json').exists(), '프로젝트 루트에서 열어주세요.'
sys.path.insert(0, str(ROOT))
from rag_baselines.data import read_json, read_jsonl, file_sha256, write_json, write_jsonl
import pandas as pd
pd.set_option('display.max_colwidth', 90)
CONFIG = read_json(ROOT / 'config/baselines.json')
print('Workspace:', ROOT)

In [ ]:
SPLIT = 'dev'                   # 설정 고정 후 'eval'
QUESTIONS_PER_TASK = 5          # None이면 해당 split 전체
CORPUS_SIZE = None              # None이면 N0; 10000/40000 등 manifest의 크기
REPEATS = 1                    # 본 지연 평가 초기 권장값 3
RETRIEVE_K = 50
EVIDENCE_K = 5
manifest = read_json(ROOT / 'data/prepared/manifest.json')
corpus_size = CORPUS_SIZE or manifest['n0']
corpus_entry = manifest['corpora'][str(corpus_size)]
assert file_sha256(ROOT / corpus_entry['path']) == corpus_entry['sha256']
all_questions = read_jsonl(ROOT / f'data/prepared/questions_{SPLIT}.jsonl')
questions = []
for task in CONFIG['qa']['tasks']:
    rows = [q for q in all_questions if q['task'] == task]
    questions.extend(rows if QUESTIONS_PER_TASK is None else rows[:QUESTIONS_PER_TASK])
labels = read_jsonl(ROOT / f'data/prepared/labels_{SPLIT}.jsonl')
display(pd.DataFrame(questions).groupby('task').size().rename('questions').to_frame())

### 2. 오프라인 모델·인덱스 준비

In [ ]:
from rag_baselines.models import Baselines
from rag_baselines.evaluation import (evaluate_cached_quality, measure_retrieval,
                                     result_table, runtime_manifest, save_run)
engine = Baselines(ROOT, retrieve_k=RETRIEVE_K, evidence_k=EVIDENCE_K)
index_info = engine.build_index(ROOT / corpus_entry['path'])
print('Device:', engine.device, 'dtype:', engine.dtype)
print('Model loading seconds:', round(engine.load_seconds, 3))
print('Index cache reused:', index_info['cache_hit'])
print('Original index build seconds:', round(index_info['build_seconds'], 3))
print('Document tokens:', index_info['document_tokens'])

### 3. 입력 잘림 검증
RetroMAE는 최대 256 토큰, DeBERTa 쌍은 최대 256 토큰이며 질문을 보존하고 문서만 자른다.
최종 LLM에 넘길 근거는 잘린 모델 입력이 아닌 **원문 passage**다.
감사용 토큰 길이 계산도 온라인 구현에 포함되므로 지연은 모델 커널만의 시간이 아니다.

In [ ]:
audit = engine.truncation_audit(read_jsonl(ROOT/'data/prepared/questions_dev.jsonl') + read_jsonl(ROOT/'data/prepared/questions_eval.jsonl'))
display(pd.DataFrame(audit).T)
engine.warmup(questions[0]['question'])

### 4. 품질 평가 — 저장된 같은 후보를 A/B가 공유

In [ ]:
quality = evaluate_cached_quality(engine, questions, labels)
quality_table = result_table(quality)
for q in questions:
    pair = [r for r in quality if r['question_id']==q['id']]
    assert [p['id'] for p in pair[0]['candidates']] == [p['id'] for p in pair[1]['candidates']]
    assert pair[0]['candidates'][0]['dense_score'] == pair[1]['candidates'][0]['dense_score']
display(quality_table.groupby(['task','method']).agg(
    questions=('question_id','nunique'), complete_evidence_rate=('complete_evidence_hit','mean'),
    candidate_rate=('complete_candidate_hit','mean')).round(3))
quality_meta = runtime_manifest(ROOT, engine)
quality_meta.update({'mode':'cached_quality', 'split':SPLIT, 'truncation_audit':audit,
                     'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
quality_directory = save_run(ROOT, quality, quality_meta, '02_measure_retrieval.ipynb')
print('Quality saved:', quality_directory)

### 5. 지연 측정 — 실제 검색을 매번 실행
단일 요청, 준비된 GPU, 워밍업 제외. 방법 순서를 seed로 섞고 CUDA를 단계마다 동기화한다.
B가 A의 검색 시간을 공짜로 재사용하지 않는다. 실제 dense 후보 일치도 확인한다.

In [ ]:
online = measure_retrieval(engine, questions, labels, repeats=REPEATS, seed=CONFIG['seed'], progress=None)
online_table = result_table(online)
for repeat in range(REPEATS):
    for q in questions:
        pair = [r for r in online if r['question_id']==q['id'] and r['repeat']==repeat]
        assert [p['id'] for p in pair[0]['candidates']] == [p['id'] for p in pair[1]['candidates']]
display(online_table.groupby(['task','method']).agg(
    questions=('question_id','nunique'), p50_ms=('retrieval_total_ms','median'),
    p95_ms=('retrieval_total_ms',lambda x:x.quantile(.95)),
    complete_evidence_rate=('complete_evidence_hit','mean')).round(3))
online_meta = runtime_manifest(ROOT, engine)
online_meta.update({'mode':'online_retrieval','split':SPLIT,'repeats':REPEATS,'truncation_audit':audit,
                    'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
online_directory = save_run(ROOT, online, online_meta, '02_measure_retrieval.ipynb')
print('Online saved:', online_directory)

### 6. 지연 분포와 모델 단계 처리량

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize':(8,4), 'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})
colors = {'A':'#2666a7','B':'#bd7900'}
fig, ax = plt.subplots()
for position, method in enumerate(('A','B')):
    values = online_table.loc[online_table.method==method,'retrieval_total_ms'].to_numpy()
    ax.scatter([position]*len(values), values, c=colors[method], marker='o' if method=='A' else 's', alpha=.65)
    ax.plot([position-.15,position+.15], [pd.Series(values).median()]*2, color='black', linewidth=2)
ax.set_xticks([0,1], ['A: dense','B: dense + rerank'])
ax.set_ylabel('Warm retrieval latency (ms)')
ax.set_title(f'A/B retrieval — {len(questions)} {SPLIT} questions, {corpus_size:,} passages, {REPEATS} repeat(s)')
ax.set_ylim(bottom=0)
ax.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(online_directory/'latency.png',dpi=150)
plt.show()
tokens = []
for method, group in online_table.groupby('method'):
    tokens.append({'method':method, 'query_tokens_per_stage_second': group.retriever_input_tokens.sum()/(group.query_encode_ms.sum()/1000),
                  'rerank_pairs_per_stage_second': (len(group)*min(RETRIEVE_K,corpus_size))/(group.rerank_ms.sum()/1000) if method=='B' else None,
                  'rerank_tokens_per_stage_second':group.reranker_input_tokens.sum()/(group.rerank_ms.sum()/1000) if method=='B' else None})
throughput_table = pd.DataFrame(tokens)
throughput_table.to_csv(online_directory/'model_throughput.csv',index=False)
display(throughput_table.round(1))

### 7. 실제 반환된 원문과 프롬프트 검수
`complete_evidence_hit`는 **어느 한 유효한 annotation의 모든 근거 passage**를 확보했는지다.
HotpotQA는 두 문서 모두 필요하다. NQ는 정답 span 기준이며, 답의 진실성이나 최종 QA 정확도와 같은 지표가 아니다.

In [ ]:
from rag_baselines.generation import build_prompt
preview = online[0]
engine.store.validate_evidence(preview)
display(pd.DataFrame([{'id':p['id'],'title':p['title'],'original_text':p['content']} for p in preview['evidence']]))
generation_config = read_json(ROOT/'config/generation.json')
prompt_previews = [{**build_prompt(r,encoding_name=generation_config['encoding_name']),
                    'question_id':r['question_id'],'method':r['method']} for r in online]
write_jsonl(online_directory/'prompt_previews.jsonl', prompt_previews)
write_json(online_directory/'truncation_audit.json', audit)
print('Common generator:', generation_config['model'], '| actual API calls: 0')
print('API usage/cost: unmeasured. Local model tokens are not API billing tokens.')

### 해석과 다음 단계
위 출력은 실행 검증 표본이다. 정확도·속도의 일반적 우열을 주장하지 않는다.
설정 고정 후 전체 eval, 세 corpus 규모, 반복 측정을 진행한다.
공통 생성과 진짜 온라인 전체 시간은 `03_measure_system_api.ipynb`에서 측정한다.